## Facebook Ego Network — Dataset and Pipeline Overview

### About the Dataset

The Facebook Ego Network dataset (McAuley & Leskovec, NIPS 2012, 
sourced from the SNAP repository) represents a snapshot of friendship 
connections extracted from Facebook, centered on a set of "ego" users 
and their local social neighborhoods.

- **Nodes** represent individual Facebook users.
- **Edges** represent an undirected friendship connection between two 
  users (if user A and user B are Facebook friends, there is an edge 
  between them).

The dataset used here contains 4,039 vertices and 88,234 edges after 
filtering to the largest connected component, with an average degree 
of roughly 44 — a dense, socially clustered network with strong 
community structure (distinct friend groups, workplaces, schools), 
which makes it a natural candidate for community-detection-based 
gamma selection rather than the diameter-endpoint approach used on 
more spatially-extended graphs like Power Grid or Oregon Router.

### What This Notebook Does

This notebook runs the full white-noise-sampling-on-graphs pipeline 
(Phases 0–7) on the Facebook Ego dataset, followed by the two-level 
Monte Carlo extension:

1. **Load and clean the graph** — parse the edge list, check 
   connectivity, filter to the largest connected component if needed.
2. **Build core matrices** — adjacency $A$, degree $D$, graph 
   Laplacian $L$; compute $\lambda_{\min}$ and the shifted Laplacian 
   $L_\sigma$ needed for a stable, invertible white-noise solve.
3. **Select $\Gamma_{\text{in}}$ / $\Gamma_{\text{out}}$** via greedy 
   modularity maximization: detect communities, take a high-degree 
   (top 5%) subset of the two largest detected communities as the 
   source and sink respectively — chosen because this dataset's 
   dominant structure is modular (distinct communities), not 
   geometric or hub-and-spoke.
4. **Run the Monte Carlo pipeline** (Phases 3–6): draw white noise, 
   solve for a spatially correlated permeability field, solve the 
   resulting Darcy flow boundary value problem, and extract the 
   quantity of interest $Q$ (total flux through the sink).
5. **Run single-level Monte Carlo** as a baseline, and the **two-level 
   Monte Carlo** extension (aggregating the graph into a smaller 
   coarse graph, then combining cheap coarse-only samples with a 
   smaller number of expensive paired fine/coarse samples) to compare 
   sample efficiency, precision, and wall-clock time between the two 
   approaches — including a cross-check using a collaborator's general 
   multi-level Monte Carlo framework.

In [10]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from functions_v2 import *
from two_level_mc import *
from graph_mlmc_model import GraphTwoLevelModel
from mlmc_runner import MLMCRunner
from sksparse.cholmod import cholesky as sparse_cholesky

In [12]:
edges, n_vertices, weights = load_graph(r"../../data/raw/facebook_combined.txt")
print(f"n_vertices={n_vertices}, edges={len(edges)}")

✓ Loaded: ../../data/raw/facebook_combined.txt
  Vertices : 4039
  Edges    : 88234
  Weighted : no

n_vertices=4039, edges=88234


In [14]:
# 3. Phase 1
A, D, L = build_graph_matrices(edges, n_vertices)

# 4. Phase 2 — sparse from the start, given graph size
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)
print("L_sigma type:", type(L_sigma))

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 4039 x 4039
  Degree range: [1, 1045]
  Non-zeros in L: 180507

✓ lambda_min = 0.000837
  (eigenvalues found: [0.         0.00083651])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse

L_sigma type: <class 'scipy.sparse._csc.csc_matrix'>


In [16]:
G_nx = nx.Graph()
G_nx.add_edges_from(edges)

In [20]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

G_nx = nx.Graph()
G_nx.add_edges_from(edges)

communities = list(greedy_modularity_communities(G_nx))
communities_sorted = sorted(communities, key=len, reverse=True)

print(f"Found {len(communities_sorted)} communities")
print(f"Sizes (top 5): {[len(c) for c in communities_sorted[:5]]}")

def top_degree_subset(community, G_nx, fraction=0.05):
    community_list = list(community)
    degrees = [(v, G_nx.degree(v)) for v in community_list]
    degrees_sorted = sorted(degrees, key=lambda x: x[1], reverse=True)
    n_keep = max(1, int(len(community_list) * fraction))
    return [v for v, d in degrees_sorted[:n_keep]]

gamma_in = top_degree_subset(communities_sorted[0], G_nx, fraction=0.05)
gamma_out = top_degree_subset(communities_sorted[1], G_nx, fraction=0.05)

print(f"gamma_in: {len(gamma_in)}, gamma_out: {len(gamma_out)}")
boundary_fraction = (len(gamma_in) + len(gamma_out)) / n_vertices
print(f"boundary fraction: {boundary_fraction:.4f}")

Found 13 communities
Sizes (top 5): [983, 815, 548, 543, 372]
gamma_in: 49, gamma_out: 40
boundary fraction: 0.0220


In [22]:
setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

gamma_in: 49, gamma_out: 40, n_vertices: 4039
Boundary fraction: 2.2035%
  -> Untested range -- proceed but verify interior_coarse is non-empty and substantial after aggregation.
n_coarse: 1995 (1995 aggregates)
Size distribution -- min: 1, max: 10, mean: 2.02
Singletons: 40 (2.0%)
gamma_in_coarse: 25, gamma_out_coarse: 19
Overlap (must be empty): set()
Interior coarse vertices: 1951 (97.8%)
coarse edges: 47965 (from 88234 fine edges)


# Single-level MLMC

In [24]:
import time

t0 = time.time()
Q_samples = monte_carlo_loop_tqdm(L_sigma, lambda_min, edges, n_vertices,
                                    gamma_in, gamma_out, N=1000, debug=False)
single_level_time = time.time() - t0

print(f"Single-level time: {single_level_time:.2f}s")
print(f"Mean Q: {Q_samples.mean():.6f}")
print(f"Std Q: {Q_samples.std():.6f}")

Monte Carlo: 100%|██████████| 1000/1000 [05:58<00:00,  2.79sample/s, mean Q=799.9426]


✓ Done — 1000 samples
  Mean Q : 799.942643
  Std Q  : 403.094774
Single-level time: 361.08s
Mean Q: 799.942643
Std Q: 403.094774


# My two-level Monte carlo

In [31]:
import time

t0 = time.time()
Q_samples = monte_carlo_loop_tqdm(L_sigma, lambda_min, edges, n_vertices,
                                    gamma_in, gamma_out, N=1000, debug=False)
single_level_time = time.time() - t0

print(f"Single-level time: {single_level_time:.2f}s")
print(f"Mean Q: {Q_samples.mean():.6f}")
print(f"Std Q: {Q_samples.std():.6f}")

Monte Carlo: 100%|██████████| 1000/1000 [04:00<00:00,  4.15sample/s, mean Q=802.7548]


✓ Done — 1000 samples
  Mean Q : 802.754773
  Std Q  : 419.698293
Single-level time: 241.41s
Mean Q: 802.754773
Std Q: 419.698293


In [6]:
import time

In [27]:
t0 = time.time()
result = run_paired_validation(setup, N=300)
paired_time = time.time() - t0

t0 = time.time()
Q_coarse_only_samples = np.array([
    run_coarse_only_sample(setup, seed=100000+n) for n in range(2000)
])
coarse_time = time.time() - t0

estimate = two_level_estimate(setup, result, N_coarse_only=2000)
own_code_time = paired_time + coarse_time

print(f"\nYour code — time: {own_code_time:.2f}s")
print(f"Estimate: {estimate['estimate']:.6f}")
print(f"Correlation: {result['correlation']:.4f}, Variance reduction: {result['variance_reduction']:.2f}x")

Paired samples: 100%|██████████| 300/300 [04:29<00:00,  1.11sample/s, Q_fine=802.6707, Q_coarse=889.2121]



N = 300 paired samples
Q_fine   : mean=802.670709  var=189806.994039
Q_coarse : mean=889.212066  var=232993.395834
Q_fine - Q_coarse : mean=-86.541357  var=2211.549845
Correlation(Q_fine, Q_coarse): 1.0000
Variance reduction: 85.83x


Coarse-only samples: 100%|██████████| 2000/2000 [09:51<00:00,  3.38sample/s, Q_coarse=876.9732]


Coarse-only base estimate (N=2000): 876.973235
Correction term mean (paired samples): -86.541357
Two-level estimate of E[Q_fine]: 790.431878
Direct fine-only mean (for comparison): 802.670709

Your code — time: 908.75s
Estimate: 790.431878
Correlation: 1.0000, Variance reduction: 85.83x


In [30]:
import numpy as np

# From your paired run
paired_diff_var = result['diff_samples'].var()
N_paired = 300  # adjust to whatever N you actually used

se_correction = np.sqrt(paired_diff_var / N_paired)

# From your coarse-only samples (need the actual array, not just the mean)
N_coarse_only = 2000  # adjust to whatever N you actually used
coarse_only_var = Q_coarse_only_samples.var()  # the array from your coarse-only loop

se_coarse = np.sqrt(coarse_only_var / N_coarse_only)

combined_se = np.sqrt(se_correction**2 + se_coarse**2)

print(f"SE from correction term: {se_correction:.6f}")
print(f"SE from coarse-only term: {se_coarse:.6f}")
print(f"Combined two-level SE (your code): {combined_se:.6f}")

SE from correction term: 2.715112
SE from coarse-only term: 9.515156
Combined two-level SE (your code): 9.894950


# Team's monte carlo

In [46]:
from graph_mlmc_model import GraphTwoLevelModel
from mlmc_runner import MLMCRunner

model = GraphTwoLevelModel(setup)
runner = MLMCRunner(model, base_seed=0)

t0 = time.time()
mlmc_result = runner.run_fixed(samples_per_level=[2000, 300])
mlmc_time = time.time() - t0

print(f"\nMLMCRunner — time: {mlmc_time:.2f}s")
print(f"Estimate: {mlmc_result.estimate:.6f}")
print(f"SE: {mlmc_result.standard_error:.6f}")


MLMCRunner — time: 854.13s
Estimate: 782.079350
SE: 9.884535
